In [3]:
def recursive_character_text_splitter(
    text,
    chunk_size=100,
    chunk_overlap=20
):
    """
    Dummy version of RecursiveCharacterTextSplitter.

    Heuristics:
    1. Try paragraph boundaries first.
    2. If still too large, try line boundaries.
    3. If still too large, try word boundaries.
    4. Finally fall back to character-level splitting.
    """

    separators = ["\n\n", "\n", " "]

    def split_text(text, separator_index=0):
        text = text.strip()

        if len(text) <= chunk_size:
            return [text]

        if separator_index >= len(separators):
            step = max(1, chunk_size - chunk_overlap)
            return [
                text[i:i + chunk_size]
                for i in range(0, len(text), step)
            ]

        separator = separators[separator_index]
        parts = text.split(separator)

        chunks = []
        current = ""

        for part in parts:
            part = part.strip()

            if not part:
                continue

            candidate = (
                part
                if not current
                else current + separator + part
            )

            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.extend(
                        split_text(current, separator_index + 1)
                    )

                current = part

        if current:
            chunks.extend(
                split_text(current, separator_index + 1)
            )

        return chunks

    chunks = split_text(text)

    # Word-based overlap
    final_chunks = []

    for i, chunk in enumerate(chunks):
        if i > 0 and chunk_overlap > 0:
            previous_words = chunks[i - 1].split()

            overlap_words = []
            length = 0

            for word in reversed(previous_words):
                if length + len(word) + 1 > chunk_overlap:
                    break
                overlap_words.insert(0, word)
                length += len(word) + 1

            if overlap_words:
                chunk = " ".join(overlap_words) + " " + chunk

        final_chunks.append(chunk)

    return final_chunks

In [4]:
sample_text = """
Machine learning is a branch of artificial intelligence.

It allows computers to learn patterns from data.

Deep learning uses neural networks with multiple layers.

Large language models are trained on massive amounts of text.
"""

chunks = recursive_character_text_splitter(
    sample_text,
    chunk_size=80,
    chunk_overlap=20
)

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}:")
    print(chunk)
    print("-" * 50)

Chunk 1:
Machine learning is a branch of artificial intelligence.
--------------------------------------------------
Chunk 2:
intelligence. It allows computers to learn patterns from data.
--------------------------------------------------
Chunk 3:
patterns from data. Deep learning uses neural networks with multiple layers.
--------------------------------------------------
Chunk 4:
multiple layers. Large language models are trained on massive amounts of text.
--------------------------------------------------
